# Common Libraries

In [1]:
import os, sys, shutil, mediapy, copy
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.gridspec import GridSpec
from ipywidgets import interact, IntSlider, FloatSlider, Layout, Button, Output, VBox, HBox

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from video_tool import save_video

# Params

In [16]:
myoarm_path = "/home/seojin/biomechanics_tools/myo_sim/arm/myoarm.xml"
mujoco_model_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_emg/Myosuite/scaled_model/scaled_model_cvt3.xml"
# mujoco_model_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_emg/Myosuite/scaled_model/scaled_model_mocap_optimized.xml"
# mujoco_model_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_41_mp/temp_cvt3.xml"

simulation_interval = 0.005

# Initialize model

In [17]:
myoarm_model = mujoco.MjModel.from_xml_path(myoarm_path)
myoarm_model.opt.timestep = simulation_interval
myoarm_data = mujoco.MjData(myoarm_model)
mujoco.mj_forward(myoarm_model, myoarm_data)

mujoco_model = mujoco.MjModel.from_xml_path(mujoco_model_path)
mujoco_model.opt.timestep = simulation_interval
mujoco_data = mujoco.MjData(mujoco_model)
mujoco.mj_forward(mujoco_model, mujoco_data)

# Render

In [18]:
scene_option = mujoco.MjvOption()
scene_option.frame = mujoco.mjtFrame.mjFRAME_WORLD

# Renderer
myoarm_renderer = mujoco.Renderer(myoarm_model, height=400, width=600)
mujoco_renderer = mujoco.Renderer(mujoco_model, height=400, width=600)

# Camera
myoarm_camera = mujoco.MjvCamera()
myoarm_camera.lookat = [0, 0, 1]
myoarm_camera.distance = 2.0

mujoco_camera = mujoco.MjvCamera()
mujoco_camera.azimuth = 180
mujoco_camera.elevation = 0
mujoco_camera.distance = 1.8

# Simulation with maximum force on specific muscle

In [19]:
# Params
is_video = True
fps = 1
n_step = 300
muscle_name = "INFSP"

# Constants
myoarm_act_id = mujoco.mj_name2id(myoarm_model, mujoco.mjtObj.mjOBJ_ACTUATOR, muscle_name)
mujoco_act_id = mujoco.mj_name2id(mujoco_model, mujoco.mjtObj.mjOBJ_ACTUATOR, muscle_name)

# Initialization
myoarm_data.qpos[:] = np.zeros_like(myoarm_data.qpos)
mujoco_data.qpos[:] = np.zeros_like(mujoco_data.qpos)
mujoco.mj_forward(myoarm_model, myoarm_data)
mujoco.mj_forward(mujoco_model, mujoco_data)

# Dummy
frames = []

# UI
output_area = Output()
apply_btn = Button(description = "Apply", button_style = "success")
def make_frame():
    myoarm_renderer.update_scene(
        myoarm_data,
        camera=myoarm_camera,
        scene_option=scene_option
    )

    mujoco_renderer.update_scene(
        mujoco_data,
        camera=mujoco_camera,
        scene_option=scene_option
    )
    img1 = myoarm_renderer.render().copy()
    img2 = mujoco_renderer.render().copy()
    if img1.shape[0] != img2.shape[0]:
        h = min(img1.shape[0], img2.shape[0])
        img1 = img1[:h]
        img2 = img2[:h]
    frame = np.concatenate([img1, img2], axis=1)
    return frame
    
def show_frame():
    frame = copy.deepcopy(make_frame())
    if is_video:
        frames.append(frame)
    else:
        with output_area:
            output_area.clear_output(wait=True)
            plt.figure(figsize=(8, 6))
            plt.imshow(frame)
            plt.axis("off")
            plt.tight_layout()
            plt.show()

for _ in range(n_step):
    myoarm_data.ctrl[myoarm_act_id] = 1.0
    mujoco.mj_step(myoarm_model, myoarm_data)

    mujoco_data.ctrl[mujoco_act_id] = 1.0
    mujoco.mj_step(mujoco_model, mujoco_data)
    show_frame()


In [20]:
mediapy.show_video(frames)

This browser does not support the video tag.


In [37]:
save_video(np.array(frames), fps = 30, output_path = "/mnt/ext1/seojin/temp/ctrl_sim_mp.mp4", is_progress_bar = True)

MoviePy - Building video /mnt/ext1/seojin/temp/ctrl_sim_mp.mp4.
MoviePy - Writing video /mnt/ext1/seojin/temp/ctrl_sim_mp.mp4



MoviePy - Done !
MoviePy - video ready /mnt/ext1/seojin/temp/ctrl_sim_mp.mp4
save: /mnt/ext1/seojin/temp/ctrl_sim_mp.mp4
